# Notebook 03: Session Strategy and Prompt Bundle
## Goal: DecisionResult → SessionStrategy + PromptBundle

This notebook takes the output of Notebook 02 (DecisionResult) and produces:

1. **SessionStrategy** — What to do next, in what order, with what guardrails
2. **PromptBundle** — The actual text blocks for the next interaction

**Design principles:**
1. Decision state drives everything
2. Questions are pre-built by NB02 — NB03 sequences and formats them
3. Internal vs external is a hard boundary
4. Max 3 questions per turn
5. Branches presented neutrally, with trade-offs

**Contract:** see `notebooks/03_session_strategy_contract.md`

In [ ]:
import json
from typing import List, Dict, Any, Optional, Literal
from dataclasses import dataclass, asdict, field
from datetime import datetime
from enum import IntEnum
import uuid

# =============================================================================
# SECTION 0: Imports from NB02 (DecisionResult shape)
# =============================================================================
# In production these come from a shared module.
# Re-declaring the DecisionResult shape so the notebook is self-contained.

@dataclass
class DecisionResult:
    packet_id: str
    current_stage: str
    hard_blockers: List[str]
    soft_blockers: List[str]
    contradictions: List[Dict[str, Any]]
    decision_state: str
    follow_up_questions: List[Dict[str, Any]]
    branch_options: List[Dict[str, Any]]
    rationale: Dict[str, Any]
    confidence_score: float

# Minimal CanonicalPacket (NB01 output) for context
@dataclass
class Slot:
    value: Any = None
    confidence: float = 0.0
    authority_level: str = "unknown"
    extraction_mode: str = "unknown"
    evidence_refs: List[Any] = field(default_factory=list)
    notes: Optional[str] = None

@dataclass
class CanonicalPacket:
    packet_id: str
    created_at: str
    last_updated: str
    facts: Dict[str, Slot] = field(default_factory=dict)
    derived_signals: Dict[str, Slot] = field(default_factory=dict)
    hypotheses: Dict[str, Slot] = field(default_factory=dict)
    unknowns: List[Any] = field(default_factory=list)
    contradictions: List[Dict[str, Any]] = field(default_factory=list)
    stage: str = "discovery"

# Question ordering by constraint impact
QUESTION_CONSTRAINT_ORDER = [
    "traveler_composition",
    "traveler_count",
    "destination_city",
    "origin_city",
    "travel_dates",
    "budget_range",
    "trip_purpose",
    "traveler_preferences",
    "selected_destinations",
    "selected_itinerary",
    "activity_preferences",
    "accommodation_type",
    "special_requests",
    "dietary_restrictions",
    "traveler_details",
    "payment_method",
]

## SessionStrategy and PromptBundle Models

In [ ]:
# =============================================================================
# SECTION 1: OUTPUT MODELS
# =============================================================================

@dataclass
class SessionStrategy:
    session_goal: str
    priority_sequence: List[str]
    tonal_guardrails: List[str]
    risk_flags: List[str]
    suggested_opening: str
    exit_criteria: List[str]
    next_action: str
    assumptions: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class PromptBundle:
    system_context: str
    user_message: str
    follow_up_sequence: List[Dict[str, Any]]
    branch_prompts: List[Dict[str, Any]]
    internal_notes: str
    constraints: List[str]

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class SessionOutput:
    """Complete output of the NB03 pipeline."""
    strategy: SessionStrategy
    prompts: PromptBundle
    decision_state: str
    confidence_score: float

    def to_dict(self) -> Dict[str, Any]:
        return {
            "decision_state": self.decision_state,
            "confidence_score": self.confidence_score,
            "strategy": self.strategy.to_dict(),
            "prompts": self.prompts.to_dict(),
        }

## Confidence-to-Tone Scaling

In [ ]:
# =============================================================================
# SECTION 2: CONFIDENCE → TONE MAPPING
# =============================================================================

def confidence_tone(confidence: float) -> str:
    if confidence < 0.3:
        return "cautious"
    elif confidence < 0.6:
        return "measured"
    elif confidence < 0.8:
        return "confident"
    else:
        return "direct"


TONE_GUARDRAILS = {
    "cautious": [
        "Acknowledge uncertainty explicitly",
        "Ask more questions than usual",
        "Do not make recommendations until more info is gathered",
        "Use tentative language: 'could be', 'might be', 'we'd need to confirm'",
    ],
    "measured": [
        "Ask focused questions",
        "Note assumptions clearly",
        "Provide limited suggestions pending confirmation",
    ],
    "confident": [
        "Proceed with draft or proposal",
        "Note remaining gaps but don't over-hedge",
        "Make specific recommendations",
    ],
    "direct": [
        "Produce final output with no hedging",
        "No mention of unknowns or hypotheses",
        "Present as a finished proposal",
    ],
}

## Question Sequencing

Implements the 5 rules from the contract:
1. Constraint-first ordering
2. Group by topic
3. Use hypothesis hints when available
4. Never more than 3 questions per turn
5. Context-aware re-asking

In [ ]:
# =============================================================================
# SECTION 3: QUESTION SEQUENCING
# =============================================================================

def sequence_questions(
    follow_up_questions: List[Dict[str, Any]],
    hypotheses: Dict[str, Any] = None,
    max_per_turn: int = 3,
    previous_questions: List[str] = None,
) -> List[Dict[str, Any]]:
    """
    Order and format questions according to sequencing rules.

    Rule 1: Constraint-first (traveler_composition → destination → origin → dates → budget → ...)
    Rule 3: Use hypothesis hints when available
    Rule 4: Max 3 questions per turn
    Rule 5: Context-aware re-asking (if previous_questions provided)

    Returns: Ordered list of question dicts ready for PromptBundle.
    """
    hypotheses = hypotheses or {}
    previous_questions = previous_questions or []

    # Sort by constraint order
    def constraint_index(q):
        field = q.get("field_name", "")
        try:
            return QUESTION_CONSTRAINT_ORDER.index(field)
        except ValueError:
            return len(QUESTION_CONSTRAINT_ORDER)  # Unknown fields go last

    sorted_questions = sorted(follow_up_questions, key=constraint_index)

    # Rule 3: Enhance with hypothesis hints
    enhanced = []
    for q in sorted_questions:
        enhanced_q = dict(q)
        field = q.get("field_name", "")

        # Check if a hypothesis exists for this field
        if field in hypotheses:
            hyp = hypotheses[field]
            hyp_value = hyp.get("value") if isinstance(hyp, dict) else getattr(hyp, 'value', None)
            hyp_conf = hyp.get("confidence", 0) if isinstance(hyp, dict) else getattr(hyp, 'confidence', 0)
            if hyp_value and hyp_conf > 0:
                enhanced_q["suggested_values"] = [hyp_value]
                enhanced_q["can_infer"] = True
                enhanced_q["inference_confidence"] = round(hyp_conf * 0.3, 2)

        enhanced.append(enhanced_q)

    # Rule 4: Max 3 per turn
    return enhanced[:max_per_turn]

## Branch Prompt Builder

Implements the 5 branch presentation rules from the contract:
1. Max 3 branches
2. Neutral framing
3. Trade-off summary per branch
4. One question per branch
5. Record the choice

In [ ]:
# =============================================================================
# SECTION 4: BRANCH PROMPT BUILDER
# =============================================================================

def build_branch_prompts(
    branch_options: List[Dict[str, Any]],
    max_branches: int = 3,
) -> List[Dict[str, Any]]:
    """
    Build prompts for branch presentation.

    Rule 1: Maximum 3 branches
    Rule 2: Neutral framing (Option A, Option B, etc.)
    Rule 3: Trade-off summary per branch
    Rule 4: One question per branch
    """
    if not branch_options:
        return []

    # Rule 1: Cap at max_branches
    capped = branch_options[:max_branches]
    prompts = []

    for i, branch in enumerate(capped):
        label = branch.get("label", f"Option {chr(65 + i)}")
        description = branch.get("description", "")

        prompt = {
            "branch_id": f"branch_{i}",
            "label": f"Option {chr(65 + i)}: {label}",
            "description": description,
            "trade_offs": _extract_trade_offs(branch),
            "question": "Does this option feel closest to what you have in mind?",
            "action": "Record traveler's choice as explicit_user authority.",
        }
        prompts.append(prompt)

    # If more branches were capped, add a note
    if len(branch_options) > max_branches:
        prompts.append({
            "branch_id": "clarification",
            "label": "Need to narrow",
            "description": f"There are {len(branch_options)} possible interpretations. Let's narrow to the top {max_branches}.",
            "trade_offs": [],
            "question": "Can you tell me more about what matters most to you so I can focus on the best options?",
            "action": "Use response to narrow branch options.",
        })

    return prompts


def _extract_trade_offs(branch: Dict[str, Any]) -> List[str]:
    """Extract trade-offs from a branch option."""
    trade_offs = []
    contradictions = branch.get("contradictions", [])
    for c in contradictions:
        values = c.get("values", [])
        if len(values) >= 2:
            trade_offs.append(f"{values[0]} vs {values[1]}")
    if not trade_offs and branch.get("description"):
        trade_offs.append(branch["description"])
    return trade_offs

## NB02 Gap Mitigations

The 3 known limitations from NB02 scenario testing:
1. Ambiguous values not detected → detect "or" / "maybe" patterns
2. No urgency handling → detect dates within 7 days
3. Budget stretch not structured → parse stretch signals

In [ ]:
# =============================================================================
# SECTION 5: NB02 GAP MITIGATIONS
# =============================================================================
import re

def detect_ambiguous_values(packet: CanonicalPacket) -> List[Dict[str, str]]:
    """
    Mitigation for NB02 Gap 1: Ambiguous values not detected.
    If a destination value contains 'or' or 'maybe', flag it.
    """
    ambiguities = []
    for field_name in ["destination_city", "origin_city", "destination", "origin"]:
        slot = packet.facts.get(field_name)
        if slot and isinstance(slot.value, str):
            value_lower = slot.value.lower()
            if " or " in value_lower or "maybe" in value_lower or "might" in value_lower:
                ambiguities.append({
                    "field": field_name,
                    "value": slot.value,
                    "reason": f"Value '{slot.value}' suggests ambiguity, not a firm decision.",
                    "action": "Add a clarification question even if NB02 said PROCEED.",
                })
    return ambiguities


def detect_urgency(packet: CanonicalPacket) -> bool:
    """
    Mitigation for NB02 Gap 2: No urgency handling.
    If travel_dates contain 'this weekend' or dates within 7 days, flag as urgent.
    """
    date_fields = ["travel_dates", "date_window", "dates"]
    urgent_patterns = [
        r"this\s+weekend", r"next\s+week", r"tomorrow", r"today",
        r"this\s+week", r"urgent", r"asap", r"immediately",
    ]

    for field_name in date_fields:
        slot = packet.facts.get(field_name)
        if slot and isinstance(slot.value, str):
            value_lower = slot.value.lower()
            if any(re.search(p, value_lower) for p in urgent_patterns):
                return True
            # Check if dates are within 7 days
            try:
                from datetime import timedelta
                # Try to parse date patterns like "2026-04-15"
                date_match = re.search(r'(\d{4}-\d{2}-\d{2})', slot.value)
                if date_match:
                    from datetime import date as date_type
                    target_date = date_type.fromisoformat(date_match.group(1))
                    today = date_type.today()
                    if 0 <= (target_date - today).days <= 7:
                        return True
            except (ValueError, ImportError):
                pass

    return False


def detect_budget_stretch(packet: CanonicalPacket) -> Optional[Dict[str, Any]]:
    """
    Mitigation for NB02 Gap 3: Budget stretch not structured.
    Parse value string for stretch signals like 'can stretch', 'flexible', 'around'.
    """
    budget_fields = ["budget_range", "budget", "budget_total"]
    stretch_patterns = [
        r"can\s+stretch", r"flexible", r"around", r"approx",
        r"roughly", r"about", r"up\s+to", r"can\s+go\s+higher",
    ]

    for field_name in budget_fields:
        slot = packet.facts.get(field_name)
        if slot and isinstance(slot.value, str):
            value_lower = slot.value.lower()
            for pattern in stretch_patterns:
                if re.search(pattern, value_lower):
                    return {
                        "field": field_name,
                        "base_value": slot.value,
                        "stretch": True,
                        "signal": re.search(pattern, value_lower).group(),
                        "action": "Ask if budget can be adjusted upward for better options.",
                    }

    return None

## Strategy Builders by Decision State

Each of the 5 decision states gets its own builder.

In [ ]:
# =============================================================================
# SECTION 6: STRATEGY BUILDERS
# =============================================================================

def _build_known_facts_summary(packet: CanonicalPacket) -> str:
    """Summarize known facts for context."""
    if not packet.facts:
        return "No facts known yet."
    parts = []
    for key, slot in packet.facts.items():
        if slot.value is not None:
            parts.append(f"{key}: {slot.value}")
    return "; ".join(parts) if parts else "No facts known yet."


def _build_internal_notes(decision: DecisionResult, packet: CanonicalPacket) -> str:
    """Build internal notes for the agent."""
    notes = []
    if decision.hard_blockers:
        notes.append(f"Hard blockers: {', '.join(decision.hard_blockers)}")
    if decision.soft_blockers:
        notes.append(f"Soft blockers: {', '.join(decision.soft_blockers)}")
    if decision.contradictions:
        notes.append(f"Active contradictions: {len(decision.contradictions)}")
        for c in decision.contradictions:
            notes.append(f"  - {c.get('field_name', '?')}: {c.get('values', [])}")
    if packet.hypotheses:
        notes.append(f"Active hypotheses: {len(packet.hypotheses)}")
    return "\n".join(notes) if notes else "No issues."


# --- ASK_FOLLOWUP ---

def build_ask_followup_strategy(
    decision: DecisionResult, packet: CanonicalPacket,
) -> SessionStrategy:
    tone = confidence_tone(decision.confidence_score)
    guardrails = list(TONE_GUARDRAILS[tone])

    # Build priority sequence from follow-up questions
    questions = sequence_questions(
        decision.follow_up_questions,
        hypotheses={k: {"value": v.value, "confidence": v.confidence}
                    for k, v in packet.hypotheses.items()},
        max_per_turn=3,
    )

    priority_sequence = [q["question"] for q in questions[:3]]

    # Suggested opening
    known = _build_known_facts_summary(packet)
    opening = f"I'd like to help plan your trip. I have some details ({known}), but I need a few more things to give you the best options."

    # Risk flags
    risks = []
    ambiguities = detect_ambiguous_values(packet)
    for amb in ambiguities:
        risks.append(f"Ambiguous {amb['field']}: '{amb['value']}'")
    if detect_urgency(packet):
        risks.append("URGENT: Travel dates within 7 days — prioritize speed.")
    stretch = detect_budget_stretch(packet)
    if stretch:
        risks.append(f"Budget has stretch signal: '{stretch['signal']}'")

    return SessionStrategy(
        session_goal=f"Collect {len(decision.hard_blockers)} blocking field(s): {', '.join(decision.hard_blockers)}.",
        priority_sequence=priority_sequence,
        tonal_guardrails=guardrails,
        risk_flags=risks,
        suggested_opening=opening,
        exit_criteria=[f"All {len(decision.hard_blockers)} hard blockers resolved"],
        next_action="ASK_FOLLOWUP",
    )


def build_ask_followup_prompts(
    decision: DecisionResult, packet: CanonicalPacket,
) -> PromptBundle:
    tone = confidence_tone(decision.confidence_score)
    known = _build_known_facts_summary(packet)

    questions = sequence_questions(
        decision.follow_up_questions,
        hypotheses={k: {"value": v.value, "confidence": v.confidence}
                    for k, v in packet.hypotheses.items()},
        max_per_turn=3,
    )

    # Build user message
    user_parts = []
    for i, q in enumerate(questions, 1):
        user_parts.append(f"{i}. {q['question']}")
    user_message = "\n".join(user_parts)

    # Build follow-up sequence
    follow_up_seq = []
    for q in questions:
        follow_up_seq.append({
            "question": q["question"],
            "field_name": q.get("field_name", ""),
            "priority": q.get("priority", "critical"),
            "trigger": "immediate",
            "suggested_values": q.get("suggested_values", []),
            "max_retries": 2,
        })

    system_context = (
        f"You are a travel planning assistant.\n\n"
        f"Context:\n"
        f"- Stage: {decision.current_stage}\n"
        f"- Decision: ASK_FOLLOWUP\n"
        f"- Known: {known}\n"
        f"- Missing: {', '.join(decision.hard_blockers)}\n\n"
        f"Your task:\n"
        f"- Ask the following questions in order\n"
        f"- Do not proceed to drafting until answers are received\n"
        f"- Do not make assumptions about missing fields\n"
        f"- Tone: {tone}"
    )

    return PromptBundle(
        system_context=system_context,
        user_message=user_message,
        follow_up_sequence=follow_up_seq,
        branch_prompts=[],
        internal_notes=_build_internal_notes(decision, packet),
        constraints=[
            "Do not proceed to drafting",
            "Do not make assumptions about missing fields",
            "Ask questions in the order provided",
        ],
    )


# --- BRANCH_OPTIONS ---

def build_branch_strategy(
    decision: DecisionResult, packet: CanonicalPacket,
) -> SessionStrategy:
    tone = "measured"  # Always measured for branches
    n_branches = min(len(decision.branch_options), 3)

    opening = "I see a couple of different ways we could approach this. Let me share the options so you can choose what fits best."

    priority_sequence = [
        "Explain that multiple valid interpretations exist.",
        "Present each branch option with trade-offs.",
        "Ask: 'Which of these feels closest to what you have in mind?'",
        "Once chosen, proceed with normal flow for that path.",
    ]

    return SessionStrategy(
        session_goal=f"Present {n_branches} alternative paths and get the traveler's preference.",
        priority_sequence=priority_sequence,
        tonal_guardrails=TONE_GUARDRAILS[tone] + [
            "Present branches neutrally — no recommendations",
            "Use 'Option A', 'Option B' — not 'Recommended' vs 'Alternative'",
        ],
        risk_flags=[f"{len(decision.contradictions)} contradiction(s) driving branch options"],
        suggested_opening=opening,
        exit_criteria=["Traveler has chosen a preferred branch"],
        next_action="BRANCH_OPTIONS",
    )


def build_branch_prompts_fn(
    decision: DecisionResult, packet: CanonicalPacket,
) -> PromptBundle:
    branch_prompts = build_branch_prompts(decision.branch_options, max_branches=3)

    branch_descriptions = []
    for bp in branch_prompts:
        branch_descriptions.append(f"\n{bp['label']}: {bp['description']}")
        for t in bp.get("trade_offs", []):
            branch_descriptions.append(f"  Trade-off: {t}")

    user_message = (
        "I see a couple of different ways we could approach this:\n\n"
        + "\n".join(branch_descriptions)
        + "\n\nWhich of these feels closest to what you have in mind?"
    )

    known = _build_known_facts_summary(packet)
    system_context = (
        f"You are a travel planning assistant.\n\n"
        f"Context:\n"
        f"- Stage: {decision.current_stage}\n"
        f"- Decision: BRANCH_OPTIONS\n"
        f"- Known: {known}\n"
        f"- Contradictions: {len(decision.contradictions)}\n\n"
        f"Your task:\n"
        f"- Present options neutrally\n"
        f"- Do not push any option\n"
        f"- Record the traveler's choice"
    )

    return PromptBundle(
        system_context=system_context,
        user_message=user_message,
        follow_up_sequence=[],
        branch_prompts=branch_prompts,
        internal_notes=_build_internal_notes(decision, packet),
        constraints=[
            "Do not recommend any option",
            "Present all options neutrally",
            "Record which option the traveler chooses",
        ],
    )


# --- PROCEED_INTERNAL_DRAFT ---

def build_internal_draft_strategy(
    decision: DecisionResult, packet: CanonicalPacket,
) -> SessionStrategy:
    tone = confidence_tone(decision.confidence_score)
    assumptions = [f"Assuming {b} will be clarified later" for b in decision.soft_blockers]

    opening = "I've put together a working draft based on what we know so far. This is for your review only — not ready for the traveler yet."

    return SessionStrategy(
        session_goal="Generate an internal draft for agent review. NOT for traveler.",
        priority_sequence=[
            "Acknowledge soft blockers that remain.",
            "Generate draft with explicit assumptions documented.",
            "Flag assumptions that could significantly change the output.",
            "Recommend next steps for resolution.",
        ],
        tonal_guardrails=TONE_GUARDRAILS[tone],
        risk_flags=[f"{len(decision.soft_blockers)} soft blocker(s): {', '.join(decision.soft_blockers)}"],
        suggested_opening=opening,
        exit_criteria=["Internal draft generated with all assumptions documented"],
        next_action="PROCEED_INTERNAL_DRAFT",
        assumptions=assumptions,
    )


def build_internal_draft_prompts(
    decision: DecisionResult, packet: CanonicalPacket,
) -> PromptBundle:
    known = _build_known_facts_summary(packet)
    assumptions = [f"Assuming {b} will be clarified later" for b in decision.soft_blockers]

    system_context = (
        f"You are a travel planning assistant.\n\n"
        f"Context:\n"
        f"- Stage: {decision.current_stage}\n"
        f"- Decision: PROCEED_INTERNAL_DRAFT\n"
        f"- Known: {known}\n"
        f"- Soft blockers: {', '.join(decision.soft_blockers)}\n\n"
        f"Your task:\n"
        f"- Generate an INTERNAL DRAFT\n"
        f"- Mark as NOT FOR TRAVELER\n"
        f"- List all assumptions:\n"
        + "\n".join(f"  - {a}" for a in assumptions)
    )

    user_message = (
        f"Create a draft proposal based on the known facts.\n\n"
        f"Known: {known}\n\n"
        f"IMPORTANT: This is an internal working draft. Do NOT share with the traveler.\n"
        f"Assumptions:\n" + "\n".join(f"- {a}" for a in assumptions)
    )

    return PromptBundle(
        system_context=system_context,
        user_message=user_message,
        follow_up_sequence=[],
        branch_prompts=[],
        internal_notes=_build_internal_notes(decision, packet),
        constraints=[
            "Do NOT present this to the traveler",
            "This is a working draft only",
            "Document all assumptions explicitly",
        ],
    )


# --- PROCEED_TRAVELER_SAFE ---

def build_traveler_safe_strategy(
    decision: DecisionResult, packet: CanonicalPacket,
) -> SessionStrategy:
    tone = "direct"

    # Mitigation: check for ambiguous values even if NB02 said proceed
    ambiguities = detect_ambiguous_values(packet)
    urgency = detect_urgency(packet)

    opening = "I've put together a complete proposal based on everything we've discussed. I'm confident this is a great fit for you."

    risks = []
    for amb in ambiguities:
        risks.append(f"Ambiguous {amb['field']}: '{amb['value']}' — should confirm before booking.")
    if urgency:
        risks.append("URGENT: Travel within 7 days — expedite booking.")

    return SessionStrategy(
        session_goal="Present a complete, traveler-ready proposal.",
        priority_sequence=[
            "Generate the proposal based on all known facts.",
            "Ground all claims in facts.",
            "Present with confidence.",
        ],
        tonal_guardrails=TONE_GUARDRAILS[tone],
        risk_flags=risks,
        suggested_opening=opening,
        exit_criteria=["Proposal presented to traveler"],
        next_action="PROCEED_TRAVELER_SAFE",
    )


def build_traveler_safe_prompts(
    decision: DecisionResult, packet: CanonicalPacket,
) -> PromptBundle:
    known = _build_known_facts_summary(packet)

    system_context = (
        f"You are a travel planning assistant.\n\n"
        f"Context:\n"
        f"- Stage: {decision.current_stage}\n"
        f"- Decision: PROCEED_TRAVELER_SAFE\n"
        f"- Known: {known}\n"
        f"- Confidence: {decision.confidence_score}\n\n"
        f"Your task:\n"
        f"- Generate a traveler-ready proposal\n"
        f"- All claims must be grounded in facts\n"
        f"- Do not mention unknowns, hypotheses, or contradictions\n"
        f"- Present with confidence"
    )

    user_message = (
        f"Create a complete proposal for the traveler.\n\n"
        f"Known facts: {known}\n\n"
        f"This is ready to share with the traveler. Present it confidently."
    )

    # Mitigation: add ambiguity questions if found
    ambiguities = detect_ambiguous_values(packet)
    follow_up_seq = []
    if ambiguities:
        for amb in ambiguities:
            follow_up_seq.append({
                "question": f"Just to confirm — did you mean '{amb['value']}' or were you still deciding?",
                "field_name": amb["field"],
                "priority": "high",
                "trigger": "after proposal presented",
                "suggested_values": [],
                "max_retries": 1,
            })

    return PromptBundle(
        system_context=system_context,
        user_message=user_message,
        follow_up_sequence=follow_up_seq,
        branch_prompts=[],
        internal_notes=_build_internal_notes(decision, packet),
        constraints=[
            "Do not mention unknowns, hypotheses, or contradictions",
            "All claims must be grounded in facts",
            "Present as a finished proposal",
        ],
    )


# --- STOP_NEEDS_REVIEW ---

def build_stop_strategy(
    decision: DecisionResult, packet: CanonicalPacket,
) -> SessionStrategy:
    opening = "This situation requires human review. Here's a summary of the issues for the agency owner."

    return SessionStrategy(
        session_goal="Escalate to human review. Generate a review briefing.",
        priority_sequence=[
            "Explain why automated processing cannot continue.",
            "Summarize the critical issues.",
            "Provide context for the human reviewer.",
            "Suggest resolution options if possible.",
        ],
        tonal_guardrails=[
            "Be specific about the issue",
            "Do not contact traveler until resolved",
            "Provide full context for the reviewer",
        ],
        risk_flags=[f"{len(decision.contradictions)} critical contradiction(s) require human judgment"],
        suggested_opening=opening,
        exit_criteria=["Human reviewer has been briefed"],
        next_action="STOP_NEEDS_REVIEW",
    )


def build_stop_prompts(
    decision: DecisionResult, packet: CanonicalPacket,
) -> PromptBundle:
    known = _build_known_facts_summary(packet)

    contradiction_summary = []
    for c in decision.contradictions:
        contradiction_summary.append(
            f"- {c.get('field_name', '?')}: {c.get('values', [])} "
            f"(sources: {c.get('sources', [])})"
        )

    system_context = (
        f"You are generating a human review briefing.\n\n"
        f"Context:\n"
        f"- Stage: {decision.current_stage}\n"
        f"- Decision: STOP_NEEDS_REVIEW\n"
        f"- Known: {known}\n\n"
        f"Contradictions:\n"
        + "\n".join(contradiction_summary)
        + "\n\n"
        f"Your task:\n"
        f"- Be specific about the issue\n"
        f"- Provide full context\n"
        f"- This is for internal review only"
    )

    user_message = (
        f"REVIEW REQUIRED\n\n"
        f"Automated processing cannot continue due to critical contradictions.\n\n"
        f"Known: {known}\n\n"
        f"Issues:\n" + "\n".join(contradiction_summary) + "\n\n"
        f"Please review and provide guidance before proceeding."
    )

    return PromptBundle(
        system_context=system_context,
        user_message=user_message,
        follow_up_sequence=[],
        branch_prompts=[],
        internal_notes=_build_internal_notes(decision, packet),
        constraints=[
            "Do not contact traveler until resolved",
            "This is for internal review only",
        ],
    )

## Main Pipeline: `build_session_output`

In [ ]:
# =============================================================================
# SECTION 7: MAIN PIPELINE
# =============================================================================

STATE_BUILDERS = {
    "ASK_FOLLOWUP": (build_ask_followup_strategy, build_ask_followup_prompts),
    "BRANCH_OPTIONS": (build_branch_strategy, build_branch_prompts_fn),
    "PROCEED_INTERNAL_DRAFT": (build_internal_draft_strategy, build_internal_draft_prompts),
    "PROCEED_TRAVELER_SAFE": (build_traveler_safe_strategy, build_traveler_safe_prompts),
    "STOP_NEEDS_REVIEW": (build_stop_strategy, build_stop_prompts),
}


def build_session_output(
    decision: DecisionResult,
    packet: CanonicalPacket,
) -> SessionOutput:
    """
    Main entry point: DecisionResult + CanonicalPacket → SessionStrategy + PromptBundle.
    """
    builders = STATE_BUILDERS.get(decision.decision_state)
    if not builders:
        raise ValueError(f"Unknown decision_state: {decision.decision_state}")

    strategy_fn, prompt_fn = builders
    strategy = strategy_fn(decision, packet)
    prompts = prompt_fn(decision, packet)

    return SessionOutput(
        strategy=strategy,
        prompts=prompts,
        decision_state=decision.decision_state,
        confidence_score=decision.confidence_score,
    )

## Tests

In [ ]:
# =============================================================================
# SECTION 8: TESTS
# =============================================================================

# Helper to create DecisionResults for testing
def make_decision(decision_state, hard_blockers=None, soft_blockers=None,
                  follow_up_questions=None, branch_options=None,
                  contradictions=None, confidence=0.7, stage="discovery"):
    return DecisionResult(
        packet_id="test",
        current_stage=stage,
        hard_blockers=hard_blockers or [],
        soft_blockers=soft_blockers or [],
        contradictions=contradictions or [],
        decision_state=decision_state,
        follow_up_questions=follow_up_questions or [],
        branch_options=branch_options or [],
        rationale={"hard_blockers": hard_blockers or [], "confidence": confidence},
        confidence_score=confidence,
    )

def make_packet(facts=None, hypotheses=None):
    facts = facts or {}
    hypotheses = hypotheses or {}
    return CanonicalPacket(
        packet_id="test_pkt",
        created_at=datetime.now().isoformat(),
        last_updated=datetime.now().isoformat(),
        facts=facts,
        hypotheses=hypotheses,
    )

_passed = 0
_failed = 0
_errors = 0

def test(name, fn):
    global _passed, _failed, _errors
    try:
        fn()
        _passed += 1
        print(f"  PASS: {name}")
    except AssertionError as e:
        _failed += 1
        print(f"  FAIL: {name}")
        print(f"        → {e}")
    except Exception as e:
        _errors += 1
        print(f"  ERR!: {name}")
        print(f"        → {type(e).__name__}: {e}")

### Test 1: ASK_FOLLOWUP → Correct question sequence

In [ ]:
# Test 1: ASK_FOLLOWUP → correct question sequence (constraint-first)
def t_ask_sequence():
    decision = make_decision("ASK_FOLLOWUP",
        hard_blockers=["destination_city", "travel_dates", "budget_range"],
        follow_up_questions=[
            {"field_name": "budget_range", "question": "What's your budget?", "priority": "critical"},
            {"field_name": "destination_city", "question": "Where to?", "priority": "critical"},
            {"field_name": "travel_dates", "question": "When?", "priority": "critical"},
        ])
    packet = make_packet()
    result = build_session_output(decision, packet)

    assert result.decision_state == "ASK_FOLLOWUP"
    # Questions should be sequenced: destination first (higher constraint), then dates, then budget
    q_fields = [q["field_name"] for q in result.prompts.follow_up_sequence]
    # destination_city comes before travel_dates and budget_range in constraint order
    if "destination_city" in q_fields:
        dest_idx = q_fields.index("destination_city")
        if "budget_range" in q_fields:
            budget_idx = q_fields.index("budget_range")
            assert dest_idx < budget_idx, f"destination ({dest_idx}) should come before budget ({budget_idx})"

test("ASK_FOLLOWUP → questions sequenced constraint-first", t_ask_sequence)

### Test 2: ASK_FOLLOWUP → Hypothesis hints used

In [ ]:
# Test 2: ASK_FOLLOWUP → hypothesis hints used
def t_ask_hypothesis_hints():
    decision = make_decision("ASK_FOLLOWUP",
        hard_blockers=["destination_city"],
        follow_up_questions=[
            {"field_name": "destination_city", "question": "Where would you like to go?", "priority": "critical"},
        ])
    packet = make_packet(hypotheses={
        "destination_city": Slot(value="Singapore", confidence=0.5, authority_level="soft_hypothesis"),
    })
    result = build_session_output(decision, packet)

    # Follow-up should have suggested_values from hypothesis
    assert len(result.prompts.follow_up_sequence) > 0
    q = result.prompts.follow_up_sequence[0]
    assert q.get("suggested_values") == ["Singapore"], \
        f"Should have hypothesis hint, got {q.get('suggested_values')}"
    assert q.get("can_infer") is True

test("ASK_FOLLOWUP → hypothesis hints used as suggested values", t_ask_hypothesis_hints)

### Test 3: BRANCH_OPTIONS → Neutral branch presentation

In [ ]:
# Test 3: BRANCH_OPTIONS → neutral branch presentation
def t_branch_neutral():
    decision = make_decision("BRANCH_OPTIONS",
        branch_options=[{
            "label": "Budget-tier options",
            "description": "Different budget interpretations suggest different trip tiers.",
            "contradictions": [{"field_name": "budget_range", "values": ["budget", "premium"]}],
        }])
    packet = make_packet()
    result = build_session_output(decision, packet)

    assert result.decision_state == "BRANCH_OPTIONS"
    assert len(result.prompts.branch_prompts) == 1
    bp = result.prompts.branch_prompts[0]
    assert "Option A" in bp["label"]
    assert len(bp["trade_offs"]) > 0

test("BRANCH_OPTIONS → neutral branch presentation with trade-offs", t_branch_neutral)

### Test 4: BRANCH_OPTIONS → Max 3 branches

In [ ]:
# Test 4: BRANCH_OPTIONS → max 3 branches, ask to narrow if more
def t_branch_max_3():
    decision = make_decision("BRANCH_OPTIONS",
        branch_options=[
            {"label": f"Option {i}", "description": f"Interpretation {i}", "contradictions": []}
            for i in range(5)
        ])
    packet = make_packet()
    result = build_session_output(decision, packet)

    # Should present 3 branches + 1 clarification prompt
    assert len(result.prompts.branch_prompts) == 4  # 3 options + 1 clarification
    clarification = result.prompts.branch_prompts[-1]
    assert clarification["branch_id"] == "clarification"
    assert "narrow" in clarification["description"].lower()

test("BRANCH_OPTIONS → max 3 branches + clarification prompt", t_branch_max_3)

### Test 5: PROCEED_INTERNAL_DRAFT → Assumptions documented

In [ ]:
# Test 5: PROCEED_INTERNAL_DRAFT → assumptions documented
def t_internal_draft_assumptions():
    decision = make_decision("PROCEED_INTERNAL_DRAFT",
        soft_blockers=["budget_range", "trip_purpose"])
    packet = make_packet()
    result = build_session_output(decision, packet)

    assert result.decision_state == "PROCEED_INTERNAL_DRAFT"
    assert len(result.strategy.assumptions) == 2
    assert any("budget_range" in a for a in result.strategy.assumptions)
    assert any("trip_purpose" in a for a in result.strategy.assumptions)
    # Constraints should include internal-only flag
    assert any("NOT" in c.upper() or "internal" in c.lower() for c in result.prompts.constraints)

test("PROCEED_INTERNAL_DRAFT → assumptions documented", t_internal_draft_assumptions)

### Test 6: PROCEED_TRAVELER_SAFE → Grounded in facts

In [ ]:
# Test 6: PROCEED_TRAVELER_SAFE → grounded in facts, no unknowns mentioned
def t_traveler_safe_grounded():
    decision = make_decision("PROCEED_TRAVELER_SAFE",
        confidence=0.85)
    packet = make_packet(facts={
        "destination_city": Slot(value="Singapore", confidence=0.95, authority_level="explicit_user"),
        "origin_city": Slot(value="Bangalore", confidence=0.95, authority_level="explicit_user"),
        "travel_dates": Slot(value="2026-03-15", confidence=0.9, authority_level="explicit_user"),
        "traveler_count": Slot(value=3, confidence=0.95, authority_level="explicit_user"),
    })
    result = build_session_output(decision, packet)

    assert result.decision_state == "PROCEED_TRAVELER_SAFE"
    # Constraints should prohibit mentioning unknowns/hypotheses
    assert any("unknown" in c.lower() for c in result.prompts.constraints)
    assert any("hypothesis" in c.lower() for c in result.prompts.constraints)
    # User message should NOT contain hedge words
    um = result.prompts.user_message.lower()
    assert "don't know" not in um
    assert "not sure" not in um
    # No assumptions for traveler-safe
    assert result.strategy.assumptions == []

test("PROCEED_TRAVELER_SAFE → grounded in facts, no unknowns", t_traveler_safe_grounded)

### Test 7: STOP_NEEDS_REVIEW → Review briefing

In [ ]:
# Test 7: STOP_NEEDS_REVIEW → review briefing generated
def t_stop_review():
    decision = make_decision("STOP_NEEDS_REVIEW",
        contradictions=[
            {"field_name": "travel_dates", "values": ["2026-03-15", "2026-04-01"],
             "sources": ["env1", "env2"]},
        ])
    packet = make_packet()
    result = build_session_output(decision, packet)

    assert result.decision_state == "STOP_NEEDS_REVIEW"
    assert "REVIEW REQUIRED" in result.prompts.user_message
    assert any("Do not contact traveler" in c for c in result.prompts.constraints)

test("STOP_NEEDS_REVIEW → review briefing generated", t_stop_review)

### Test 8: Question sequencing → Constraint-first order

In [ ]:
# Test 8: Question sequencing → constraint-first order (all 4 hard blockers)
def t_sequence_constraint_first():
    questions = [
        {"field_name": "budget_range", "question": "Budget?", "priority": "critical"},
        {"field_name": "travel_dates", "question": "When?", "priority": "critical"},
        {"field_name": "origin_city", "question": "From where?", "priority": "critical"},
        {"field_name": "destination_city", "question": "Where to?", "priority": "critical"},
        {"field_name": "traveler_count", "question": "How many?", "priority": "critical"},
    ]
    result = sequence_questions(questions, max_per_turn=10)
    fields = [q["field_name"] for q in result]

    # traveler_count (composition) should be first
    assert fields[0] == "traveler_count", f"First should be traveler_count, got {fields[0]}"
    # destination_city should be before origin_city, dates, budget
    dest_idx = fields.index("destination_city")
    origin_idx = fields.index("origin_city")
    dates_idx = fields.index("travel_dates")
    budget_idx = fields.index("budget_range")
    assert dest_idx < origin_idx, f"destination before origin: {fields}"
    assert dest_idx < dates_idx, f"destination before dates: {fields}"
    assert dest_idx < budget_idx, f"destination before budget: {fields}"

test("Question sequencing → constraint-first order", t_sequence_constraint_first)

### Test 9: Question sequencing → Max 3 per turn

In [ ]:
# Test 9: Question sequencing → max 3 per turn
def t_sequence_max_3():
    questions = [
        {"field_name": f"field_{i}", "question": f"Q{i}", "priority": "critical"}
        for i in range(5)
    ]
    result = sequence_questions(questions, max_per_turn=3)
    assert len(result) == 3, f"Should return max 3, got {len(result)}"

test("Question sequencing → max 3 per turn", t_sequence_max_3)

### Test 10: Confidence → Tone scaling

In [ ]:
# Test 10: Confidence → tone scaling
def t_tone_scaling():
    assert confidence_tone(0.1) == "cautious"
    assert confidence_tone(0.25) == "cautious"
    assert confidence_tone(0.35) == "measured"
    assert confidence_tone(0.55) == "measured"
    assert confidence_tone(0.65) == "confident"
    assert confidence_tone(0.75) == "confident"
    assert confidence_tone(0.85) == "direct"
    assert confidence_tone(0.99) == "direct"

    # Verify guardrails exist for each tone
    for tone in ["cautious", "measured", "confident", "direct"]:
        assert tone in TONE_GUARDRAILS
        assert len(TONE_GUARDRAILS[tone]) > 0

test("Confidence → tone scaling correct", t_tone_scaling)

### Test 11: Gap Mitigation — Ambiguous values detected

In [ ]:
# Test 11: Gap mitigation — ambiguous values detected
def t_gap_ambiguous():
    packet = make_packet(facts={
        "destination_city": Slot(value="Andaman or Sri Lanka", confidence=0.6, authority_level="explicit_owner"),
    })
    ambiguities = detect_ambiguous_values(packet)
    assert len(ambiguities) == 1
    assert ambiguities[0]["field"] == "destination_city"
    assert "or" in ambiguities[0]["value"].lower()

    # Test with "maybe"
    packet2 = make_packet(facts={
        "destination_city": Slot(value="maybe Thailand", confidence=0.5, authority_level="explicit_owner"),
    })
    ambiguities2 = detect_ambiguous_values(packet2)
    assert len(ambiguities2) == 1

test("Gap mitigation — ambiguous values detected ('or', 'maybe')", t_gap_ambiguous)

### Test 12: Gap Mitigation — Urgency detected

In [ ]:
# Test 12: Gap mitigation — urgency detected
def t_gap_urgency():
    packet = make_packet(facts={
        "travel_dates": Slot(value="this weekend", confidence=0.9, authority_level="explicit_user"),
    })
    assert detect_urgency(packet) is True

    packet2 = make_packet(facts={
        "travel_dates": Slot(value="urgent, need to book ASAP", confidence=0.9, authority_level="explicit_user"),
    })
    assert detect_urgency(packet2) is True

    # Non-urgent
    packet3 = make_packet(facts={
        "travel_dates": Slot(value="March 2026", confidence=0.8, authority_level="explicit_user"),
    })
    assert detect_urgency(packet3) is False

test("Gap mitigation — urgency detected for 'this weekend', 'ASAP'", t_gap_urgency)

### Test 13: Gap Mitigation — Budget stretch detected

In [ ]:
# Test 13: Gap mitigation — budget stretch detected
def t_gap_budget_stretch():
    packet = make_packet(facts={
        "budget_range": Slot(value="200000 (can stretch if it's good)", confidence=0.7, authority_level="explicit_owner"),
    })
    stretch = detect_budget_stretch(packet)
    assert stretch is not None
    assert stretch["stretch"] is True
    assert "can stretch" in stretch["signal"]

    # Flexible
    packet2 = make_packet(facts={
        "budget_range": Slot(value="around 3L, flexible", confidence=0.6, authority_level="explicit_owner"),
    })
    stretch2 = detect_budget_stretch(packet2)
    assert stretch2 is not None

    # No stretch signal
    packet3 = make_packet(facts={
        "budget_range": Slot(value="exactly 250000", confidence=0.9, authority_level="explicit_user"),
    })
    stretch3 = detect_budget_stretch(packet3)
    assert stretch3 is None

test("Gap mitigation — budget stretch detected", t_gap_budget_stretch)

### Test 14: PROCEED_TRAVELER_SAFE → Ambiguity follow-up added

In [ ]:
# Test 14: PROCEED_TRAVELER_SAFE → ambiguity triggers follow-up question
def t_safe_ambiguity_followup():
    decision = make_decision("PROCEED_TRAVELER_SAFE", confidence=0.85)
    packet = make_packet(facts={
        "destination_city": Slot(value="Andaman or Sri Lanka", confidence=0.6, authority_level="explicit_owner"),
        "origin_city": Slot(value="Bangalore", confidence=0.95, authority_level="explicit_user"),
        "travel_dates": Slot(value="2026-03-15", confidence=0.9, authority_level="explicit_user"),
        "traveler_count": Slot(value=3, confidence=0.95, authority_level="explicit_user"),
    })
    result = build_session_output(decision, packet)

    # Should have a follow-up question about the ambiguity
    assert len(result.prompts.follow_up_sequence) > 0
    assert "confirm" in result.prompts.follow_up_sequence[0]["question"].lower()

test("PROCEED_TRAVELER_SAFE → ambiguous value triggers follow-up", t_safe_ambiguity_followup)

### Test 15: PROCEED_TRAVELER_SAFE → Urgency triggers risk flag

In [ ]:
# Test 15: PROCEED_TRAVELER_SAFE → urgency triggers risk flag
def t_safe_urgency_risk():
    decision = make_decision("PROCEED_TRAVELER_SAFE", confidence=0.85)
    packet = make_packet(facts={
        "destination_city": Slot(value="Singapore", confidence=0.95, authority_level="explicit_user"),
        "origin_city": Slot(value="Bangalore", confidence=0.95, authority_level="explicit_user"),
        "travel_dates": Slot(value="this weekend", confidence=0.9, authority_level="explicit_user"),
        "traveler_count": Slot(value=3, confidence=0.95, authority_level="explicit_user"),
    })
    result = build_session_output(decision, packet)

    assert any("URGENT" in r for r in result.strategy.risk_flags)

test("PROCEED_TRAVELER_SAFE → urgency triggers risk flag", t_safe_urgency_risk)

## Summary

In [ ]:
# Print summary
print(f"\n{'='*60}")
print(f"  NB03 Test Summary")
print(f"{'='*60}")
print(f"  Passed:  {_passed}")
print(f"  Failed:  {_failed}")
print(f"  Errors:  {_errors}")
print(f"  Total:   {_passed + _failed + _errors}")
if _passed + _failed + _errors > 0:
    print(f"  Rate:    {_passed}/{_passed + _failed + _errors} = {_passed/(_passed+_failed+_errors)*100:.0f}%")

## What NB03 Does

| Test | What It Validates | Result |
|------|------------------|--------|
| 1 | ASK_FOLLOWUP → questions sequenced constraint-first | — |
| 2 | ASK_FOLLOWUP → hypothesis hints used | — |
| 3 | BRANCH_OPTIONS → neutral branch presentation | — |
| 4 | BRANCH_OPTIONS → max 3 branches + clarification | — |
| 5 | PROCEED_INTERNAL_DRAFT → assumptions documented | — |
| 6 | PROCEED_TRAVELER_SAFE → grounded in facts | — |
| 7 | STOP_NEEDS_REVIEW → review briefing generated | — |
| 8 | Question sequencing → constraint-first order | — |
| 9 | Question sequencing → max 3 per turn | — |
| 10 | Confidence → tone scaling | — |
| 11 | Gap: ambiguous values detected | — |
| 12 | Gap: urgency detected | — |
| 13 | Gap: budget stretch detected | — |
| 14 | PROCEED_TRAVELER_SAFE + ambiguity → follow-up | — |
| 15 | PROCEED_TRAVELER_SAFE + urgency → risk flag | — |